Before writing any sort of application, CRUD functions need to be developed and tested. This notebook provides a space for that.

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
# Running this cell REQUIRES a module named con_lib.py in this notebook's PARENT DIRECTORY
# See con_EXAMPLE.py for a template to set up con_lib.py
# ALTERNATELY, provide a connection string in a different way

import sys
sys.path.append("..") # con_lib is the parent directory of this notebook
from con_lib import connection_string

In [3]:
engine = create_engine(connection_string)

# Create

## Define

In [4]:
def create_friend(name, max_loans=2, notes=None):
    df = pd.DataFrame(
        [[name, max_loans, notes]],
        columns=["name", "max_loans", "notes"]
    )
    df.to_sql(
        "friends", 
        if_exists="append", 
        con=connection_string, 
        index=False
    )
    message = f"Added '{name}' to 'friends'."
    
    return message

In [5]:
def create_book(title, isbn, author=None, genre=None):
    df = pd.DataFrame(
        [[title, author, genre, isbn]], 
        columns=["title", "author", "genre", "isbn"]
    )
    df.to_sql(
        "books",
        if_exists="append", 
        con=connection_string, 
        index=False
    )
    message = f"Added '{title}' to 'books'."
        
    return message

In [6]:
def create_loan(friend, book, loan_date=pd.Timestamp.today().date(), next_contact=pd.Timestamp.today().date() + pd.Timedelta(30, "d"), notes=None):
    df = pd.DataFrame(
        [[book["isbn"], friend["friend_id"], loan_date, next_contact, notes]],
        columns = ["isbn", "friend_id", "loan_date", "next_contact", "notes"]
    )
    df.to_sql(
        "loans",
        if_exists="append",
        con=connection_string,
        index=False
    )
    message = f"Added '{friend["name"]}' borrowed '{book["title"]}' to 'loans'."

    return message

## Test

In [7]:
create_friend('Ed')
create_friend("Eddy", 1)
create_friend("Edd", notes="Not sure he can read")

pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso Klein,1,None
6,7,Ed,2,None
7,8,Eddy,1,None
8,9,Edd,2,Not sure he can read


In [8]:
create_book("Words on a Page", "0000000000000")
create_book("Alice's Big Nap", "0000000000", "A. Snooze"),
create_book("More Words on a Page", "1234567890", genre="Book")

pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


In [9]:
friend =  pd.read_sql("SELECT * FROM friends WHERE friend_id = 6", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9785566778899'", con=connection_string).iloc[0]
create_loan(friend, book)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 1", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780062316110'", con=connection_string).iloc[0]
loan_date = '2026-01-01'
create_loan(friend, book, loan_date)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 2", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780385490818'", con=connection_string).iloc[0]
next_contact='2026-02-15'
create_loan(friend, book, next_contact=next_contact)
             
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 5", con=connection_string).iloc[0]
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780143127741'", con=connection_string).iloc[0]
notes="Reading for thesis."
create_loan(friend, book, notes=notes)


pd.read_sql("loans", con=connection_string)

,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,1,2026-01-01,NaT,2026-03-15,None
1,9780143127741,5,2026-02-13,NaT,2026-03-15,Reading for thesis.
2,9780385490818,2,2026-02-13,NaT,2026-02-15,None
3,9780987654321,5,2025-07-02,2025-07-02,2025-07-25,None
4,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
5,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,None
6,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,None
7,9785566778899,6,2026-02-13,NaT,2026-03-15,None


# Read

## Define

In [4]:
def prettify_df(df):
    df.columns = [c.upper() if c == "isbn" else c.replace("_", " ").capitalize() for c in df.columns]
    return df.fillna("")

In [7]:
def read_friends():
    return pd.read_sql("friends", con=connection_string)

def display_friends(friends):
    return friends.pipe(prettify_df).loc[:, "Name":]

In [8]:
def read_books(available_only=False):
    books = pd.read_sql("books", con=connection_string)
    if available_only:
        loans = pd.read_sql("loans", con=connection_string)
        books = pd.merge(books, loans, on="isbn", how="left").query("friend_id.isna()")[books.columns]
    return books

def display_books(books):
    return books.pipe(prettify_df).sort_values(by="Title")

In [10]:
def read_loans():
    return pd.read_sql("loans", con=connection_string)

def display_loans():
    friends = read_friends()
    books = read_books()
    loans = read_loans()
    for c in ["loan_date", "last_contact", "next_contact"]:
        loans[c] = loans[c].dt.strftime("%Y-%m-%d")

    display_columns = ["title", "name", "loan_date", "last_contact", "next_contact", "notes"]
    df = (
        pd.merge(loans, friends, on="friend_id", suffixes=["", "_no"])
        .merge(books, on="isbn")
        [display_columns]
    )
    return df.pipe(prettify_df).sort_values(by="Loan date")

## Test

In [14]:
read_friends()
display_friends(read_friends())

,Name,Max loans,Notes
0,Ellie Martinez,2,
1,Davey,3,Gets recommendations from Ellie
2,Luca Schmidt,1,Always takes long loans
3,Amira Jansen,2,
4,Fix Bauer,3,
5,Soso Klein,1,
6,Ed,2,
7,Eddy,1,
8,Edd,2,Not sure he can read


In [15]:
read_books()
display_books(read_books())

,Title,Author,Genre,ISBN
0,Alice's Big Nap,A. Snooze,,0000000000
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
12,Gardens of Glass,Ivy Thornton,Fantasy,9785566778899
2,More Words on a Page,,Book,1234567890
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
11,The Clockmaker's Son,Hugo Vernier,Steampunk,9784455667788
10,The Paper Trail,Liane Forestier,Historical Fiction,9781234567890
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


In [11]:
read_loans()
display_loans()

,Title,Name,Loan date,Last contact,Next contact,Notes
4,The Secret Ingredient,Luca Schmidt,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
5,The Paper Trail,Ellie Martinez,2025-06-15,2025-06-15,2025-07-15,
3,Echoes of the Past,Fix Bauer,2025-07-02,2026-02-13,2026-02-20,Got caught in rainstorm with book
0,The Alchemist,Ellie Martinez,2026-01-01,,2026-03-15,
1,Sapiens: A Brief History of Humankind,Fix Bauer,2026-02-13,,2026-03-15,Reading for thesis.
2,The Poisonwood Bible,Davey,2026-02-13,,2026-02-15,


# Update

## Define

In [17]:
def format_field(field):
    if field == "dates":
        return "Contact dates"
    elif field == "isbn":
        return "ISBN"
    else:
        return field.replace("_", " ").capitalize()

In [18]:
def update_friend(friend, field, new_data):
    update_query = f"""UPDATE friends 
    SET {field} = '{new_data}' 
    WHERE friend_id = {friend["friend_id"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."

In [19]:
def update_book(book, field, new_data):
    update_query = f"""UPDATE books
    SET {field} = '{new_data}'
    WHERE isbn = {book["isbn"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."

In [20]:
def update_loan(loan, field, new_data):
    update_query = f"""UPDATE loans
    SET {field} = '{new_data}'
    WHERE isbn = {loan["isbn"]} AND friend_id = {loan["friend_id"]};"""
    if field == 'dates':
        update_query = f"""UPDATE loans
        SET last_contact = '{new_data[0]}', next_contact = '{new_data[1]}'
        WHERE isbn = {loan["isbn"]} AND friend_id = {loan["friend_id"]};"""
    with engine.begin() as connection:
        connection.execute(text(update_query))
        return f"{format_field(field)} updated."

## Test

In [21]:
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 6", con=engine).iloc[0] # Soso Klein

update_friend(friend, 'name', 'Soso')
update_friend(friend, 'max_loans', 2)
update_friend(friend, 'notes', "Doesn't answer phone, use socials.".replace("'", "\\'")) 

pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,None
7,8,Eddy,1,None
8,9,Edd,2,Not sure he can read


In [22]:
book = pd.read_sql("SELECT * FROM books WHERE isbn = '9785566778899'", con=engine).iloc[0] # Gardens of Glass

update_book(book, 'title', 'Gardens of Obsidian')
update_book(book, 'author', 'Poison Ivy')
update_book(book, 'genre', 'Romantasy')
update_book(book, 'isbn', '9988776655879')


pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


In [23]:
loan = pd.read_sql("SELECT * FROM loans WHERE isbn = '9780987654321' AND friend_id = 5", con=engine).iloc[0] # Echos of the Past

today = pd.Timestamp.today().date()
next_week = today + pd.Timedelta(1, "w")
update_loan(loan, 'notes', 'Got caught in rainstorm with book')
update_loan(loan, 'dates', (today, next_week))

pd.read_sql("loans", con=connection_string)

,isbn,friend_id,loan_date,last_contact,next_contact,notes
0,9780062316110,1,2026-01-01,NaT,2026-03-15,None
1,9780143127741,5,2026-02-13,NaT,2026-03-15,Reading for thesis.
2,9780385490818,2,2026-02-13,NaT,2026-02-15,None
3,9780987654321,5,2025-07-02,2026-02-13,2026-02-20,Got caught in rainstorm with book
4,9781122334455,3,2025-03-12,2025-06-20,2025-07-10,Promised to return after summer holidays.
5,9781234567890,1,2025-06-15,2025-06-15,2025-07-15,None
6,9784455667788,4,2025-07-01,2025-06-30,2025-07-30,None
7,9988776655879,6,2026-02-13,NaT,2026-03-15,None


# Delete

## Define

In [24]:
def delete_friend(friend):
    delete_query = f"""DELETE FROM friends
    WHERE friend_id = '{friend["friend_id"]}';"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        return f"Removed '{friend['name']}' from 'friends'."

In [25]:
def delete_book(book):
    delete_query = f"""DELETE FROM books
    WHERE isbn = '{book["isbn"]}';"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        return f"Removed '{book['title']}' from 'books'."

In [26]:
def delete_loan(loan):
    lookup_table = (pd.read_sql("loans", con=connection_string).loc[[loan.name]]
                    .merge(pd.read_sql("friends", con=connection_string), on="friend_id")
                    .merge(pd.read_sql("books", con=connection_string), on="isbn")
                    .iloc[0]
                   )
    delete_query = f"""DELETE FROM loans
    WHERE friend_id = '{lookup_table["friend_id"]}' AND isbn = '{lookup_table["isbn"]}';"""
    with engine.begin() as connection:
        connection.execute(text(delete_query))
        return f"Removed '{lookup_table["name"]}' borrowed '{lookup_table["title"]}' from 'loans'."

## Test

In [27]:
pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,None
7,8,Eddy,1,None
8,9,Edd,2,Not sure he can read


In [28]:
friend = pd.read_sql("friends", con=connection_string).iloc[-1]
delete_friend(friend)

"Removed 'Edd' from 'friends'."

In [29]:
pd.read_sql("friends", con=connection_string)

,friend_id,name,max_loans,notes
0,1,Ellie Martinez,2,None
1,2,Davey,3,Gets recommendations from Ellie
2,3,Luca Schmidt,1,Always takes long loans
3,4,Amira Jansen,2,None
4,5,Fix Bauer,3,None
5,6,Soso,2,"Doesn't answer phone, use socials."
6,7,Ed,2,None
7,8,Eddy,1,None


In [30]:
pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


In [31]:
book = pd.read_sql("books", con=connection_string).iloc[-1]
delete_book(book)

"Removed 'Gardens of Obsidian' from 'books'."

In [32]:
pd.read_sql("books", con=connection_string)

,title,author,genre,isbn
0,Alice's Big Nap,A. Snooze,None,0000000000
1,Words on a Page,None,None,0000000000000
2,More Words on a Page,None,Book,1234567890
3,The Alchemist,Paulo Coelho,Fiction,9780062316110
4,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Nonfiction,9780143127741
5,The Wind-Up Bird Chronicle,Haruki Murakami,Fiction,9780307949486
6,The Poisonwood Bible,Barbara Kingsolver,Fiction,9780385490818
7,"Thinking, Fast and Slow",Daniel Kahneman,Psychology,9780553386790
8,Echoes of the Past,Julian Marsh,Thriller,9780987654321
9,The Secret Ingredient,Samira Nouri,Romance,9781122334455


Test comparison of table before and after delete and see if the 'missing row' matches the orginally specified loan

In [33]:
table_pre = pd.read_sql("loans", con=connection_string)

In [34]:
loan = pd.read_sql("loans", con=connection_string).iloc[-1]
delete_loan(loan)

"Removed 'Amira Jansen' borrowed 'The Clockmaker's Son' from 'loans'."

In [35]:
table_post = pd.read_sql("loans", con=connection_string)

In [36]:
dropped_line = pd.concat([table_pre, table_post]).drop_duplicates(keep=False).iloc[0]
dropped_line

isbn                  9784455667788
friend_id                         4
loan_date       2025-07-01 00:00:00
last_contact    2025-06-30 00:00:00
next_contact    2025-07-30 00:00:00
notes                          None
Name: 6, dtype: object

In [37]:
loan.equals(dropped_line)

True

# Validate

## Define

In [38]:
def validate_name(name):
    if (name is None) or (not name.strip()):
        return "Warning. Empty name not accepted."
    else:
        return ""

In [39]:
def validate_isbn(isbn):
    if len(isbn) not in (10, 13):
        return "Warning. ISBN must be 10 or 13 digits long."
    elif not isbn.isnumeric():
        return "Warning. ISBN must be numeric."
    elif isbn in pd.read_sql("SELECT isbn FROM books", con=connection_string)["isbn"].values:
        return "Warning. This ISBN is already in use."
    else:
        return ""

In [40]:
def validate_title(title):
    if (title is None) or (not title.strip()):
        return "Warning. Empty title not accepted."
    else:
        return ""

In [41]:
def validate_loan_taker(friend):
    current_loans = pd.read_sql("loans", con=connection_string)
    if friend["friend_id"] in current_loans["friend_id"].unique():
        num_loans = current_loans.value_counts("friend_id").loc[friend["friend_id"]]
        if friend["max_loans"] == num_loans:
            return f"Warning. {friend["name"]} has already reached their maximum loan allowance."
    else:
        return ""

In [42]:
def validate_loan_item(book):
    current_loans = pd.read_sql("loans", con=connection_string)
    if book["isbn"] in current_loans["isbn"].values:
        return f"Warning. {book["title"]} is already on loan."
    else:
        return ""

## Test

In [43]:
validate_name("Xena")
validate_name("")
validate_name("    ")
validate_name(None)

'Warning. Empty name not accepted.'

In [44]:
validate_isbn("1111111111111")
validate_isbn("00001")
validate_isbn("000000000a")
validate_isbn("9780307949486")

'Warning. This ISBN is already in use.'

In [45]:
validate_title("This is Not a Title")
validate_title("")
validate_title("    ")
validate_title(None)

'Warning. Empty title not accepted.'

In [46]:
friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 6", con=connection_string).iloc[0]
validate_loan_taker(friend)

friend = pd.read_sql("SELECT * FROM friends WHERE friend_id = 3", con=connection_string).iloc[0]
validate_loan_taker(friend)

'Warning. Luca Schmidt has already reached their maximum loan allowance.'

In [47]:
book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9780307949486'", con=connection_string).iloc[0] # The Wind-Up Bird Chronicle 
validate_loan_item(book)

book = pd.read_sql("SELECT * FROM books WHERE ISBN = '9781122334455'", con=connection_string).iloc[0] # The Secret Ingredient
validate_loan_item(book)

'Warning. The Secret Ingredient is already on loan.'